In [ ]:
#@title # **Installation**
#@markdown ### This takes around 2 minutes
from IPython.display import clear_output
from google.colab import drive
import os

#@markdown Mount Google Drive to cache the ~5GB VoxCPM2 model between sessions
use_drive = True  #@param {type: "boolean"}

if use_drive:
    drive.mount("/content/drive", force_remount=True)
    cache_base = "/content/drive/MyDrive/ai_cache"
    os.makedirs(cache_base, exist_ok=True)
    os.environ["HF_HOME"] = os.path.join(cache_base, "huggingface")
    os.environ["MODELSCOPE_CACHE"] = os.path.join(cache_base, "modelscope")
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    os.makedirs(os.environ["MODELSCOPE_CACHE"], exist_ok=True)
    print(f"Cache set to: {cache_base}")

#@markdown **Important:** If you previously ran this notebook and hit a torch compile error,
#@markdown restart the runtime (Runtime > Restart runtime) so the fix below takes effect.

print("Installing...")

!pip install uv > /dev/null 2>&1
!git clone https://github.com/Mwimwii/lvm.git > /dev/null 2>&1
!git clone https://github.com/Mwimwii/vo.git > /dev/null 2>&1
!uv pip install -r lvm/requirements.txt > /dev/null 2>&1
!apt-get update > /dev/null 2>&1

clear_output()
print("Installation done!")
if use_drive:
    print(f"Models will be cached in: {cache_base}")


In [ ]:
#@title Run UI
import os

%cd /content/lvm

# Disable torch.compile to avoid CUDA graph TLS errors in Gradio worker threads
%env TORCH_COMPILE_DISABLE=1

# Clear any stale compiled kernels from previous runs
!rm -rf /tmp/torchinductor_root/

#@markdown The type of tunnel you wanna use for seeing the public link, so that if one of them is down, you can use the other one.
Tunnel = "Gradio" #@param ["Gradio", "Ngrok", "Cloudflare", "LocalTunnel", "Horizon"]

#@markdown Also when using Ngrok, Cloudflare, LocalTunnel or Horizon, you need to wait for the Local URL to appear, and only after that click on the Public URL which is above.

#@markdown Use the following option **only if you chose Ngrok** as the Tunnel:
#@markdown You can get the Ngrok Tunnel Authtoken here: https://dashboard.ngrok.com/tunnels/authtokens/new.
ngrok_authtoken = "" #@param {type:"string"}

# @markdown You can optionally change the Ngrok Tunnel Region to one nearer to you for lower latency
ngrok_region = "us - United States (Ohio)" # @param ["au - Australia (Sydney)","eu - Europe (Frankfurt)", "ap - Asia/Pacific (Singapore)", "us - United States (Ohio)", "jp - Japan (Tokyo)", "in - India (Mumbai)","sa - South America (Sao Paulo)"]

#@markdown Use the following option **only if you chose Horizon** as the Tunnel:
#@markdown You can get the Horizon ID here: https://hrzn.run/dashboard/ , login, on the 2nd step, it shows an hrzn login YOUR_ID, you need to copy that id.
horizon_id = "" #@param {type:"string"}

if Tunnel == "Gradio":
  share_flag = "--share"
elif Tunnel == "Ngrok":
  share_flag = ""
  from pyngrok import conf, ngrok
  NgrokConfig = conf.PyngrokConfig()
  NgrokConfig.auth_token = ngrok_authtoken
  NgrokConfig.region = ngrok_region[0:2]
  conf.set_default(NgrokConfig)
  http_tunnel = ngrok.connect(9999, bind_tls=True)
  print("Ngrok Tunnel Public URL:", http_tunnel.public_url)
elif Tunnel == "Cloudflare":
  share_flag = ""
  !curl -LO https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
  !dpkg -i cloudflared-linux-amd64.deb
  !rm -rf nohup.out
  import time
  !nohup cloudflared tunnel --url localhost:9999 &
  clear_output()
  time.sleep(5)
  cloudflare_url = !grep -oE "https://[a-zA-Z0-9.-]+\.trycloudflare\.com" nohup.out
  print(f"Cloudflare Tunnel Public URL: {cloudflare_url[0]}")
elif Tunnel == "LocalTunnel":
  share_flag = ""
  !npm install -g localtunnel
  import time
  import urllib
  with open("url.txt", "w") as file:
        file.write("")
  get_ipython().system_raw("lt --port 9999 >> url.txt 2>&1 &")
  time.sleep(4)
  endpoint_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip("\n")
  with open("url.txt", "r") as file:
      tunnel_url = file.read()
      tunnel_url = tunnel_url.replace("your url is: ", "")
  clear_output()
  print(f"LocalTunnel Tunnel Public URL: \033[0m\033[93m{tunnel_url}\033[0m", end="\033[0m")
  print(f"LocalTunnel Password: {endpoint_ip}")
elif Tunnel == "Horizon":
  share_flag = ""
  !npm install -g @hrzn/cli
  import time
  !hrzn login $horizon_id
  with open("url.txt", "w") as file:
        file.write("")
  get_ipython().system_raw("hrzn tunnel http://localhost:9999 >> url.txt 2>&1 &")
  time.sleep(4)
  with open("url.txt", "r") as file:
      tunnel_url = file.read()
      tunnel_url = !grep -oE "https://[a-zA-Z0-9.-]+\.hrzn\.run" url.txt
      tunnel_url = tunnel_url[0]
  clear_output()
  print(f"Horizon Tunnel Public URL: \033[0m\033[93m{tunnel_url}\033[0m")

# kills previously running processes
!fuser -k 9999/tcp

command = f"python app.py {share_flag}"
!{command}
